# 03 Multi-Source Asset Coverage

This multi-source notebook compares coverage, groups by implementation status, filters API-key sources, shows unsupported dataset behaviour, and includes recommendation examples for public-first workflows.


In [ ]:
from collections import defaultdict
from algotradeplan.data import DataHub

hub = DataHub()
coverage = hub.coverage_table()
source_names = hub.sources()

api_key_required = [row["Source"] for row in coverage if row["Requires API key"] == "yes"]
by_status = defaultdict(list)
for row in coverage:
    by_status[row["Implementation status"]].append(row["Source"])

recommendations = {
    "crypto_spot_kline": hub.recommend_sources("crypto_spot_kline", allow_api_key=False),
    "crypto_perp_funding": hub.recommend_sources("crypto_perp_funding", allow_api_key=False),
    "macro_indicators": hub.recommend_sources("macro_indicators", allow_api_key=False),
    "public_news": hub.recommend_sources("public_news", allow_api_key=False),
}

cg_assets = hub.discover_assets("coingecko", limit=5)
cg = hub.ingest(source="coingecko", symbol=(cg_assets[0] if cg_assets else "bitcoin"), datasets=["tick", "kline"], allow_partial=True)
stooq = hub.ingest(source="stooq", symbol="aapl.us", datasets=["kline"], allow_partial=True)
gdelt = hub.ingest(source="gdelt", symbol="bitcoin", datasets=["news"], allow_partial=True)
world_bank = hub.ingest(source="world_bank", symbol="NY.GDP.MKTP.CD", datasets=["macro"], allow_partial=True)
ecb = hub.ingest(source="ecb", symbol="EUR", datasets=["tick", "macro"], allow_partial=True)
defillama_assets = hub.discover_assets("defillama", limit=1)
defillama = hub.ingest(source="defillama", symbol=(defillama_assets[0] if defillama_assets else "aave"), datasets=["macro", "fundamentals"], allow_partial=True)

unsupported = hub.ingest(source="coingecko", symbol="bitcoin", datasets=["orderbook"], allow_partial=True)

{
    "sources": source_names,
    "coverage": coverage[:8],
    "api_key_required": api_key_required,
    "grouped_by_implementation_status": dict(by_status),
    "recommendations": recommendations,
    "sample_dataset_coverage": {
        "coingecko": cg.dataset_coverage,
        "stooq": stooq.dataset_coverage,
        "gdelt": gdelt.dataset_coverage,
        "world_bank": world_bank.dataset_coverage,
        "ecb": ecb.dataset_coverage,
        "defillama": defillama.dataset_coverage,
    },
    "unsupported": unsupported.source_issues,
    "allow_partial=True": True,
}
